## 半監督學習：核心資料稀缺策略（偽標籤與知識蒸餾）

- 目標：掌握半監督學習與模型壓縮的核心技術。利用 PyTorch 實作 高門檻偽標籤（Pseudo-labeling） 策略，將海量未標註的晶圓圖轉化為訓練養分；同時實作 知識蒸餾（Knowledge Distillation） 演算法，將大模型的缺陷辨識能力壓縮並轉移至輕量化的小模型，以順利部署於產線邊緣端。


### 1. 核心資料稀缺策略：高門檻偽標籤 (Pseudo-labeling)

- 實作：新產品引進（NPI）初期，我們只有 50 片人工精準標註的缺陷晶圓（Labeled），但產線每天會噴出上萬片未標註的晶圓圖（Unlabeled）。我們利用初期小模型對未標註資料進行預測，「只有當預測信心度（Confidence）高於 95% 時，才賦予其偽標籤（Pseudo-label）並丟回訓練集」，以此實現模型自我進化。


In [ ]:
import torch
import torch.nn as nn
import numpy as np


# 模擬初期訓練好、效能尚可的模型（教師模型原型）
class SimpleWaferNet(nn.Module):
    def __init__(self, num_classes=3):
        super(SimpleWaferNet, self).__init__()
        self.fc = nn.Linear(10, num_classes)

    def forward(self, x):
        return self.fc(x)


# 實作偽標籤提取函數
def get_pseudo_labels_v2(
    unlabeled_data: torch.Tensor, model: nn.Module, threshold: float = 0.95
):
    """對應專案：get_pseudo_labels_v2() 策略"""
    model.eval()
    with torch.no_grad():
        logits = model(unlabeled_data)
        # 使用 Softmax 計算機率分佈
        probs = torch.softmax(logits, dim=1)
        # 找出每片晶圓機率最高的類別及其信心度
        max_probs, pseudo_labels = torch.max(probs, dim=1)

        # 關鍵篩選：只有高於門檻值（例如 95%）的資料才保留
        high_confidence_mask = max_probs >= threshold

        filtered_inputs = unlabeled_data[high_confidence_mask]
        filtered_labels = pseudo_labels[high_confidence_mask]

    print(f"原始未標註資料: {len(unlabeled_data)} 片")
    print(
        f"通過 {threshold * 100}% 門檻的黃金數據: {len(filtered_inputs)} 片 (轉換為偽標籤)"
    )
    return filtered_inputs, filtered_labels


# --- 執行測試 ---
torch.manual_seed(42)
mock_unlabeled_wafers = torch.randn(20, 10)  # 模擬 20 片未標註晶圓特徵
toy_model = SimpleWaferNet(num_classes=3)

gold_inputs, gold_labels = get_pseudo_labels_v2(
    mock_unlabeled_wafers, toy_model, threshold=0.60
)  # 演示用降低門檻

### 2. 部署優化：知識蒸餾與模型壓縮 (Knowledge Distillation)

- 實作：在雲端伺服器上，我們訓練了一個巨大的 TeacherModel（例如 ResNet50），辨識率極高，但因為參數太大，產線機台端的邊緣設備（Edge Device）無法即時推理。我們透過知識蒸餾，讓輕量化的 StudentModel（如 MobileNetV2 概念）不僅學習真實標籤（Hard Label），更去學習大模型輸出的軟機率分佈（Soft Labels，包含缺陷之間的相似度雜訊）。


In [ ]:
import torch.nn.functional as F

# 定義大型教師模型與輕量化學生模型
class LargeTeacherNet(nn.Module):
    def __init__(self):
        super(LargeTeacherNet, self).__init__()
        self.net = nn.Sequential(nn.Linear(10, 128), nn.ReLU(), nn.Linear(128, 3))
    def forward(self, x): return self.net(x)

class SmallStudentNet(nn.Module):
    def __init__(self):
        super(SmallStudentNet, self).__init__()
        self.net = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 3))
    def forward(self, x): return self.net(x)

# 知識蒸餾核心損失函數 (Distillation Loss)
def distill_teacher_to_student(student_logits, teacher_logits, labels, T=3.0, alpha=0.7):
    """
    對應專案：distill_teacher_to_student() 模型壓縮
    T (Temperature): 溫度參數，用來平滑化大模型的機率分佈，使其釋出更多細節知識
    alpha: 真實標籤損失與大模型軟標籤損失的分配權重
    """
    # 傳統交叉熵損失 (Hard Label Loss)
    loss_hard = nn.CrossEntropyLoss()(student_logits, labels)
    
    # 蒸餾損失 (Soft Label KL Divergence Loss)
    # 使用溫度 T 對 Logits 進行平滑化
    soft_student = F.log_softmax(student_logits / T, dim=1)
    soft_teacher = F.softmax(teacher_logits / T, dim=1)
    # KL 散度計算兩者分佈的接近程度
    loss_soft = nn.KLDivLoss(reduction='batchmean')(soft_student, soft_teacher) * (T ** 2)
    
    # 聯合損失
    return alpha * loss_hard + (1.0 - alpha) * loss_soft

# --- 模擬訓練一摺 (One Batch Forward) ---
teacher = LargeTeacherNet()
student = SmallStudentNet()

mock_x = torch.randn(4, 10)
mock_y = torch.tensor([0, 1, 2, 1], dtype=torch.long)

# 分別前向傳播
t_logits = teacher(mock_x)
s_logits = student(mock_x)

total_loss = distill_teacher_to_student(s_logits, t_logits, mock_y, T=3.0, alpha=0.5)
print(f"知識蒸餾前向損失計算完成。聯合損失總和: {total_loss.item():.4f}")

- 總結：在半導體產線將深度學習落地的過程中，常面臨兩個現實骨頭：第一是 『初期高品質標註資料極度稀缺』，第二是 『產線機台邊緣設備運算資源薄弱』。為了打通這兩個瓶頸，我實作了半監督與模型壓縮的最佳化 Pipeline。首先，在資料層面，我編寫了 get_pseudo_labels_v2()。我們利用少量的黃金標註資料訓練出基底模型，再去對海量的未標註晶圓圖進行預測。為了防止髒 label 污染模型，我設定了高達 95% 的信心度（Confidence Threshold）防禦門檻，只將絕對純淨的預測結果轉化為偽標籤，讓模型自動擴展訓練集。其次，在部署層面，我採用了 知識蒸餾 (Knowledge Distillation) 技術。我們在伺服器端讓笨重的龐大教師模型跑出最高水準，再透過 distill_teacher_to_student 函數，利用 高溫度參數（Temperature） 將大模型對缺陷空間拓樸的『軟知識（Soft Labels）』，完美蒸餾、壓縮到只有原本不到十分之一大小的輕量化學生模型中。這讓我們的演算法能在不犧牲精準度的前提下，順利跑在產線端低功耗的邊緣計算盒上。
